In [ ]:
# Setup: DuckDB — full support for GROUPING SETS, ROLLUP, CUBE
import duckdb
import pandas as pd

con = duckdb.connect(':memory:')

def sql_executer(sql):
    con.execute(sql)

def sql_executer_printer(sql):
    print(con.execute(sql).df().fillna('ALL (Grand Total)').to_string(index=False))

sql_executer("""
CREATE TABLE server_costs (
    region VARCHAR, env VARCHAR, server_type VARCHAR, cost DOUBLE
)
""")
sql_executer("""
INSERT INTO server_costs VALUES
    ('us-east', 'prod', 'db', 1500.0),
    ('us-east', 'prod', 'web', 800.0),
    ('us-east', 'dev', 'db', 400.0),
    ('us-west', 'prod', 'db', 1600.0),
    ('us-west', 'prod', 'web', 850.0),
    ('us-west', 'dev', 'web', 300.0)
""")
print("Table: server_costs (6 rows, 3 dimensions, 1 metric)")
sql_executer_printer("SELECT * FROM server_costs LIMIT 3")

# SQL — GROUPING SETS, ROLLUP, and CUBE
**Day 10 — SQL Module**

---

<div style="padding:15px;border-left:8px solid #11998e;background:#e8faf5;border-radius:4px;">
<strong>Core Insight:</strong> Normal `GROUP BY` gives you one combination of dimensions. What if a dashboard needs the total by `region`, total by `env`, AND the grand total? You could write three queries connected by `UNION ALL`. Or, you can use advanced grouping extensions to generate sub-totals and grand totals directly inside the database engine in a single pass.
</div>

**Reminder:** `ROLLUP` produces hierarchical totals. `CUBE` produces ALL possible combinations. `GROUPING SETS` lets you explicitly specify exact combinations.

## GROUPING SETS — Surgical Aggregation

Explicitly declare strictly the combinations you want aggregated, dispensing with `UNION ALL`.
If you want:
1. Total cost by Region
2. Total cost by Environment
3. Grand Total (no dimensions)
Use `GROUPING SETS ((region), (env), ())`

In [ ]:
sql_sets = """
SELECT
    region,
    env,
    SUM(cost) AS total_cost
FROM server_costs
GROUP BY GROUPING SETS (
    (region),   -- subtotal by region
    (env),      -- subtotal by env
    ()          -- grand total
)
ORDER BY region, env
"""
print("Explicit GROUPING SETS — Region Total, Env Total, Grand Total:")
sql_executer_printer(sql_sets)
print("Notice exactly 5 rows returned. No row for (region, env). NULL implies 'ALL'.")

## ROLLUP — Hierarchical Reporting

`ROLLUP (A, B, C)` generates subtotals moving strictly from right to left.
It produces:
1. (A, B, C)
2. (A, B)
3. (A)
4. () Grand Total

Perfect for geographical or temporal hierarchies (Year -> Month -> Day).

In [ ]:
sql_rollup = """
SELECT
    region,
    env,
    SUM(cost) AS total_cost
FROM server_costs
GROUP BY ROLLUP (region, env)
ORDER BY region, env
"""
print("ROLLUP(region, env) — Hierarchical Drilling:")
print("Expect: (region, env), (region), and ()")
sql_executer_printer(sql_rollup)

## CUBE — The Cross-Tab Powerhouse

`CUBE (A, B)` generates EVERY possible combination.
It produces `2^n` combinations:
1. (A, B)
2. (A)
3. (B)
4. ()

Dangerous on massive cardinality (e.g. `CUBE(10 columns)` generates 1024 aggregations per tuple!).

In [ ]:
sql_cube = """
SELECT
    region,
    env,
    SUM(cost) AS total_cost
FROM server_costs
GROUP BY CUBE (region, env)
ORDER BY region, env
"""
print("CUBE(region, env) — Exhaustive Totals:")
sql_executer_printer(sql_cube)

## Interview Q&A & Terminology

**Q: What is the core difference between `ROLLUP(A, B)` and `CUBE(A, B)`?**<br>
A: `ROLLUP` assumes a hierarchy. It calculates `(A, B)`, `(A)`, and `Grand Total`. It explicitly does NOT calculate `(B)`. `CUBE` assumes no hierarchy and calculates all combinations, including `(B)` alone.

**Q: How does the DB represent the 'Grand Total' in the result set columns?**<br>
A: It inserts primitive `NULL` into the grouping variable columns. When displaying UI grids, we commonly wrap dimensions in `COALESCE(col, 'ALL')` to render it safely.

**Q: Why use `GROUPING SETS` instead of `UNION ALL` spanning multiple queries?**<br>
A: `UNION ALL` forces the DB optimizer to perform `N` separate full table scans or index traverses. By declaring `GROUPING SETS`, the DB Engine scans the base dataset precisely **once**, maintaining internal hash aggregators dynamically. Massive time and CPU reductions.

**The Citi Angle**
At Citi, finance dashboards required slicing infrastructure costs by Region, by Data Center, and by Application Tier simultaneously. Performing exhaustive `UNION ALL`s on 4-billion row billing reports ran for 45 minutes and occasionally broke memory thresholds. We rewrote the pipelines using `GROUP BY CUBE (region, datacenter, tier)`. By instructing the DB to perform a monolithic single pass, we dropped the execution to 6 minutes and exposed every conceivable drill-down dynamically in one table.